In [ ]:
# %% 최근 파일 5개 삭제
import os, glob
from datetime import datetime

OUTPUT_DIR = "./data/7_qwen_test"

files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.json"), recursive=True)
files_with_mtime = [(f, os.path.getmtime(f)) for f in files]
files_with_mtime.sort(key=lambda x: x[1], reverse=True)

for f, mtime in files_with_mtime[:7]:
    ts = datetime.fromtimestamp(mtime).strftime("%m-%d %H:%M:%S")
    rel = os.path.relpath(f, OUTPUT_DIR)
    os.remove(f)
    print(f"  삭제: {ts} {rel}")

print(f"\n✅ {len(files_with_mtime)}개 삭제 완료")

In [ ]:
"""
동시 텔롭 최대 개수 분석
- 6_telop_instances의 JSON에서 (start_sec, end_sec) 기반으로
  0.1초 bin별 동시 활성 인스턴스 수를 계산
"""
import os
import json
import glob
import numpy as np
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

INST_DIR = "./data/6_telop_instances"
BIN_SIZE = 0.1
NUM_WORKERS = 32


def analyze_one(json_path: str) -> dict:
    """영상 1개의 동시 텔롭 통계"""
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    if not instances:
        return {"max_simul": 0, "mean_simul": 0.0, "n_instances": 0, "path": json_path}

    # bin별 동시 활성 인스턴스 수
    bins = {}
    for inst in instances:
        s, e = inst["start_sec"], inst["end_sec"]
        b_start = int(s / BIN_SIZE)
        b_end = int(e / BIN_SIZE)
        for b in range(b_start, b_end):
            bins[b] = bins.get(b, 0) + 1

    if not bins:
        return {"max_simul": 0, "mean_simul": 0.0, "n_instances": len(instances), "path": json_path}

    counts = list(bins.values())
    return {
        "max_simul": max(counts),
        "mean_simul": np.mean(counts),
        "n_instances": len(instances),
        "path": json_path,
    }


def main():
    json_paths = sorted(glob.glob(os.path.join(INST_DIR, "**", "*.json"), recursive=True))
    print(f"📁 분석 대상: {len(json_paths):,}개 영상")

    results = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(analyze_one, p): p for p in json_paths}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="분석"):
            results.append(fut.result())

    # ── 전체 통계 ──
    max_simuls = np.array([r["max_simul"] for r in results])
    mean_simuls = np.array([r["mean_simul"] for r in results])

    print(f"\n{'='*60}")
    print(f"📊 동시 텔롭 개수 통계 ({len(results):,}개 영상)")
    print(f"{'='*60}")

    print(f"\n  영상별 최대 동시 텔롭 수:")
    print(f"    mean:   {max_simuls.mean():.1f}")
    print(f"    median: {np.median(max_simuls):.1f}")
    print(f"    p90:    {np.percentile(max_simuls, 90):.0f}")
    print(f"    p95:    {np.percentile(max_simuls, 95):.0f}")
    print(f"    p99:    {np.percentile(max_simuls, 99):.0f}")
    print(f"    max:    {max_simuls.max():.0f}")

    print(f"\n  영상별 평균 동시 텔롭 수:")
    nonzero_means = mean_simuls[mean_simuls > 0]
    print(f"    mean:   {nonzero_means.mean():.2f}")
    print(f"    median: {np.median(nonzero_means):.2f}")

    # ── 분포 ──
    print(f"\n  최대 동시 텔롭 수 분포:")
    print(f"    {'구간':<12} {'영상수':>8} {'비율':>8} {'누적':>8}")
    print(f"    {'─'*12} {'─'*8} {'─'*8} {'─'*8}")
    bins_edges = [0, 1, 2, 3, 4, 5, 6, 8, 10, 15, 20, 50, 1000]
    cumul = 0
    for i in range(len(bins_edges) - 1):
        lo, hi = bins_edges[i], bins_edges[i + 1]
        cnt = int(((max_simuls >= lo) & (max_simuls < hi)).sum())
        cumul += cnt
        label = f"{lo}" if hi - lo == 1 else f"{lo}~{hi-1}"
        print(f"    {label:<12} {cnt:>8} {cnt/len(max_simuls)*100:>7.1f}% {cumul/len(max_simuls)*100:>7.1f}%")

    # ── top 10 ──
    print(f"\n  동시 텔롭 최대인 영상 top 10:")
    top10 = sorted(results, key=lambda r: r["max_simul"], reverse=True)[:10]
    for r in top10:
        rel = os.path.relpath(r["path"], INST_DIR)
        print(f"    max={r['max_simul']:>3}  inst={r['n_instances']:>4}  {rel}")

    # ── 채널별 평균 ──
    channel_maxs = {}
    for r in results:
        ch = r["path"].split(os.sep)[-2] if os.sep in r["path"] else "unknown"
        if ch not in channel_maxs:
            channel_maxs[ch] = []
        channel_maxs[ch].append(r["max_simul"])

    ch_avg = {ch: np.mean(vs) for ch, vs in channel_maxs.items()}
    print(f"\n  채널별 평균 최대 동시 텔롭 top 10:")
    for ch, avg in sorted(ch_avg.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"    avg_max={avg:>5.1f}  ({len(channel_maxs[ch]):>3}개 영상)  {ch}")

    # ── K값 제안 ──
    print(f"\n{'='*60}")
    print(f"💡 K값 제안")
    print(f"{'='*60}")
    for p in [90, 95, 99]:
        v = np.percentile(max_simuls, p)
        cover = (max_simuls <= v).sum() / len(max_simuls) * 100
        print(f"  K={int(v):>2} → {cover:.1f}% 영상 커버 (p{p})")


if __name__ == "__main__":
    main()

In [ ]:
# %% 텔롭 인스턴스 수 분포 확인
import os
import numpy as np
import polars as pl
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

INST_DIR = "./data/6_telop_instances"


def analyze_one(args):
    channel, fname = args
    path = os.path.join(INST_DIR, channel, fname)
    try:
        df = pl.read_parquet(path, glob=False)
    except Exception:
        return None

    n_total = len(df)
    if n_total == 0:
        return {"n_total": 0, "max_concurrent": 0, "channel": channel}

    # 같은 시점 최대 공존 개수 (sweep line)
    events = []
    for row in df.iter_rows(named=True):
        events.append((float(row["start_sec"]), 1))
        events.append((float(row["end_sec"]), -1))
    events.sort(key=lambda x: (x[0], -x[1]))  # 같은 시각이면 시작이 먼저

    cur = 0
    mx = 0
    for _, d in events:
        cur += d
        mx = max(mx, cur)

    return {"n_total": n_total, "max_concurrent": mx, "channel": channel}


# 태스크 수집
tasks = []
for channel in sorted(os.listdir(INST_DIR)):
    ch_dir = os.path.join(INST_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in os.listdir(ch_dir):
        if fname.endswith(".parquet"):
            tasks.append((channel, fname))

print(f"🎯 분석 대상: {len(tasks):,}개 영상")

# 병렬 실행
results = []
with ProcessPoolExecutor(max_workers=32) as pool:
    futures = [pool.submit(analyze_one, t) for t in tasks]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="분석"):
        r = fut.result()
        if r is not None:
            results.append(r)

# 집계
totals = np.array([r["n_total"] for r in results])
concurrents = np.array([r["max_concurrent"] for r in results])

def summary(arr, name):
    print(f"\n📊 {name}")
    print(f"  mean:   {arr.mean():.1f}")
    print(f"  median: {np.median(arr):.0f}")
    print(f"  std:    {arr.std():.1f}")
    print(f"  min:    {arr.min()}")
    print(f"  max:    {arr.max()}")
    print(f"  p90:    {np.percentile(arr, 90):.0f}")
    print(f"  p95:    {np.percentile(arr, 95):.0f}")
    print(f"  p99:    {np.percentile(arr, 99):.0f}")
    print(f"  p99.9:  {np.percentile(arr, 99.9):.0f}")


summary(totals, "영상당 총 텔롭 인스턴스 수")
summary(concurrents, "영상당 동시 공존 최대 인스턴스 수")

# 구간별 분포
print(f"\n📊 영상당 총 인스턴스 수 구간 분포")
bins = [0, 10, 50, 100, 200, 500, 1000, 2000, 5000, 100000]
for lo, hi in zip(bins[:-1], bins[1:]):
    n = ((totals >= lo) & (totals < hi)).sum()
    pct = n / len(totals) * 100
    print(f"  {lo:>5}~{hi:<6}: {n:>6,} ({pct:>5.1f}%)")

print(f"\n📊 동시 공존 최대 개수 구간 분포")
bins_c = [0, 5, 10, 20, 50, 100, 200, 500, 10000]
for lo, hi in zip(bins_c[:-1], bins_c[1:]):
    n = ((concurrents >= lo) & (concurrents < hi)).sum()
    pct = n / len(concurrents) * 100
    print(f"  {lo:>4}~{hi:<6}: {n:>6,} ({pct:>5.1f}%)")

# 채널별 평균도 같이 보기 (어떤 채널이 텔롭을 많이 쓰는지)
from collections import defaultdict
by_ch = defaultdict(list)
for r in results:
    by_ch[r["channel"]].append(r["n_total"])

ch_means = [(ch, np.mean(v)) for ch, v in by_ch.items()]
ch_means.sort(key=lambda x: -x[1])
print(f"\n📊 영상 평균 인스턴스 수 top/bottom 10 채널")
print("  [TOP 10]")
for ch, m in ch_means[:10]:
    print(f"    {ch:40s} {m:7.1f}")
print("  [BOTTOM 10]")
for ch, m in ch_means[-10:]:
    print(f"    {ch:40s} {m:7.1f}")

In [1]:
# %% 데이터 용량 확인
import os
from tqdm import tqdm

def get_dir_size(path):
    total = 0
    files = []
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            files.append(os.path.join(dirpath, f))
    for fp in tqdm(files, desc=os.path.basename(path)):
        total += os.path.getsize(fp)
    return total

dirs = {
    "8_telop_position": "./data/8_telop_position",
    "4_stt_results": "./data/4_stt_results",
}
files = {
    "8_text_embeddings.pt": "./data/8_text_embeddings.pt",
}

total = 0
for name, path in dirs.items():
    if os.path.exists(path):
        size = get_dir_size(path)
        total += size
        print(f"📁 {name}: {size / 1e9:.2f} GB\n")

for name, path in files.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        total += size
        print(f"📄 {name}: {size / 1e9:.2f} GB")

print(f"\n📊 합계: {total / 1e9:.2f} GB")

8_telop_position: 100%|██████████| 66400/66400 [00:05<00:00, 12288.88it/s] 


📁 8_telop_position: 1.00 GB



4_stt_results: 100%|██████████| 66400/66400 [00:15<00:00, 4292.93it/s]

📁 4_stt_results: 0.17 GB

📄 8_text_embeddings.pt: 9.11 GB

📊 합계: 10.29 GB


In [1]:
# 확인용 셀 (단독 실행)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend

device = torch.device("cuda")

# 간단한 테스트용 attention
d_model = 256
n_heads = 8
head_dim = d_model // n_heads

q = torch.randn(4, 6400, d_model, device=device, dtype=torch.bfloat16)
k = torch.randn(4, 6400, d_model, device=device, dtype=torch.bfloat16)
v = torch.randn(4, 6400, d_model, device=device, dtype=torch.bfloat16)

q = q.reshape(4, 6400, n_heads, head_dim).transpose(1, 2)
k = k.reshape(4, 6400, n_heads, head_dim).transpose(1, 2)
v = v.reshape(4, 6400, n_heads, head_dim).transpose(1, 2)

# 1. mask 없이 (windowed self-attn의 non-shifted layer)
print("=== mask 없음 (non-shifted window) ===")
try:
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        out = F.scaled_dot_product_attention(q, k, v)
    print("✅ Flash Attention 사용 가능")
except RuntimeError as e:
    print(f"❌ Flash Attention 불가: {e}")

# 2. mask 있을 때 (shifted window)
print("\n=== mask 있음 (shifted window) ===")
mask = torch.zeros(4, n_heads, 6400, 6400, device=device, dtype=torch.bfloat16)
try:
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    print("✅ Flash Attention 사용 가능")
except RuntimeError as e:
    print(f"❌ Flash Attention 불가: {e}")

try:
    with sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    print("✅ Memory Efficient Attention 사용 가능")
except RuntimeError as e:
    print(f"❌ Memory Efficient 불가: {e}")

# 3. cross-attention (6400 query × 9 key)
print("\n=== cross-attention (6400×9) ===")
k_cross = torch.randn(4, 9, d_model, device=device, dtype=torch.bfloat16)
v_cross = torch.randn(4, 9, d_model, device=device, dtype=torch.bfloat16)
k_cross = k_cross.reshape(4, 9, n_heads, head_dim).transpose(1, 2)
v_cross = v_cross.reshape(4, 9, n_heads, head_dim).transpose(1, 2)

try:
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        out = F.scaled_dot_product_attention(q, k_cross, v_cross)
    print("✅ Flash Attention 사용 가능")
except RuntimeError as e:
    print(f"❌ Flash Attention 불가: {e}")

=== mask 없음 (non-shifted window) ===
✅ Flash Attention 사용 가능

=== mask 있음 (shifted window) ===
❌ Flash Attention 불가: No available kernel. Aborting execution.
✅ Memory Efficient Attention 사용 가능

=== cross-attention (6400×9) ===
✅ Flash Attention 사용 가능


/tmp/ipykernel_1962150/3138312088.py:36: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:915.)
  out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
/tmp/ipykernel_1962150/3138312088.py:36: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
/tmp/ipykernel_1962150/3138312088.py:36: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:917.)
  out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
/tmp/ipykernel_1962150/3138312088.py:36: UserWarning: Flash Attention does not support non-null attn_mask. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:262.)
  out = F.scaled_dot_product_attenti

In [2]:
"""
ViT Patch Mask 모델 하이퍼파라미터 결정을 위한 데이터 분포 확인
- duration / frame 수 분포
- 프레임별 동시 활성 텔롭 수 분포
- 텔롭 text_len 분포
실행: python check_data_dist.py
"""
import os, json
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

POS_DIR = "./data/8_telop_position"
NUM_WORKERS = 32
FPS = 10
GRID_W = 80
GRID_H = 80


def analyze_one(args):
    channel, path = args
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))

    n_inst = len(instances)
    if n_inst == 0:
        return {
            "duration": duration,
            "n_inst": 0,
            "max_concurrent": 0,
            "text_lens": [],
        }

    starts = np.array([inst["start_sec"] for inst in instances])
    ends = np.array([inst["end_sec"] for inst in instances])
    text_lens = [len(inst["text"]) for inst in instances]

    # 프레임별 동시 활성 텔롭 수
    n_frames = int(duration * FPS) + 1
    times = np.arange(n_frames, dtype=np.float32) / FPS
    active_matrix = (starts[None, :] <= times[:, None] + 0.05) & (ends[None, :] > times[:, None])
    concurrent_counts = active_matrix.sum(axis=1)
    max_concurrent = int(concurrent_counts.max())

    return {
        "duration": duration,
        "n_inst": n_inst,
        "max_concurrent": max_concurrent,
        "text_lens": text_lens,
    }


def main():
    # ── 파일 목록 수집 (셀 1과 동일한 방식) ──
    json_paths = []
    for channel in sorted(os.listdir(POS_DIR)):
        ch_dir = os.path.join(POS_DIR, channel)
        if not os.path.isdir(ch_dir):
            continue
        for fname in sorted(os.listdir(ch_dir)):
            if fname.endswith(".json"):
                json_paths.append((channel, os.path.join(ch_dir, fname)))

    print(f"📁 분석 대상: {len(json_paths):,}개 영상\n")

    results = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futures = {pool.submit(analyze_one, args): args for args in json_paths}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="분석"):
            results.append(fut.result())

    # ── duration / frame 수 ──
    durations = np.array([r["duration"] for r in results])
    frame_counts = (durations * FPS).astype(int) + 1

    print(f"\n{'='*60}")
    print(f"📊 duration (초)")
    print(f"  mean: {durations.mean():.1f}  median: {np.median(durations):.1f}")
    print(f"  p95:  {np.percentile(durations, 95):.1f}")
    print(f"  p99:  {np.percentile(durations, 99):.1f}")
    print(f"  max:  {durations.max():.1f}")

    print(f"\n📊 frame 수 (FPS={FPS})")
    print(f"  mean: {frame_counts.mean():.0f}  median: {np.median(frame_counts):.0f}")
    print(f"  p95:  {np.percentile(frame_counts, 95):.0f}")
    print(f"  p99:  {np.percentile(frame_counts, 99):.0f}")
    print(f"  max:  {frame_counts.max()}")

    # ── 동시 활성 텔롭 수 ──
    max_concurrents = np.array([r["max_concurrent"] for r in results])

    print(f"\n📊 프레임별 동시 활성 텔롭 수 (영상별 max)")
    print(f"  mean: {max_concurrents.mean():.1f}  median: {np.median(max_concurrents):.0f}")
    print(f"  p95:  {np.percentile(max_concurrents, 95):.0f}")
    print(f"  p99:  {np.percentile(max_concurrents, 99):.0f}")
    print(f"  max:  {max_concurrents.max()}")

    # ── 텔롭 text_len ──
    all_text_lens = []
    for r in results:
        all_text_lens.extend(r["text_lens"])
    all_text_lens = np.array(all_text_lens) if all_text_lens else np.array([0])

    print(f"\n📊 텔롭 text_len (글자 수)")
    print(f"  mean: {all_text_lens.mean():.1f}  median: {np.median(all_text_lens):.1f}")
    print(f"  p95:  {np.percentile(all_text_lens, 95):.0f}")
    print(f"  p99:  {np.percentile(all_text_lens, 99):.0f}")
    print(f"  max:  {all_text_lens.max()}")

    # ── 인스턴스 수 ──
    n_insts = np.array([r["n_inst"] for r in results])

    print(f"\n📊 영상별 인스턴스 수")
    print(f"  mean: {n_insts.mean():.1f}  median: {np.median(n_insts):.0f}")
    print(f"  p95:  {np.percentile(n_insts, 95):.0f}")
    print(f"  p99:  {np.percentile(n_insts, 99):.0f}")
    print(f"  max:  {n_insts.max()}")

    # ── 하이퍼파라미터 제안 ──
    print(f"\n{'='*60}")
    print(f"📌 하이퍼파라미터 제안 (max 기준)")
    print(f"  MAX_FRAMES   = {int(frame_counts.max())}")
    print(f"  K_TELOP_FEAT = {int(max_concurrents.max())}")
    print(f"  FEAT_DIM     = 2 + {int(max_concurrents.max())} = {2 + int(max_concurrents.max())}")
    print(f"  text_len 정규화 = / {int(all_text_lens.max())}")


if __name__ == "__main__":
    main()

📁 분석 대상: 66,400개 영상



분석: 100%|██████████| 66400/66400 [00:04<00:00, 16263.85it/s]



📊 duration (초)
  mean: 41.6  median: 40.9
  p95:  67.6
  p99:  126.8
  max:  180.2

📊 frame 수 (FPS=10)
  mean: 417  median: 410
  p95:  677
  p99:  1269
  max:  1803

📊 프레임별 동시 활성 텔롭 수 (영상별 max)
  mean: 5.8  median: 5
  p95:  14
  p99:  24
  max:  138

📊 텔롭 text_len (글자 수)
  mean: 6.4  median: 4.0
  p95:  18
  p99:  30
  max:  169

📊 영상별 인스턴스 수
  mean: 56.3  median: 29
  p95:  194
  p99:  403
  max:  4251

📌 하이퍼파라미터 제안 (max 기준)
  MAX_FRAMES   = 1803
  K_TELOP_FEAT = 138
  FEAT_DIM     = 2 + 138 = 140
  text_len 정규화 = / 169
